# Knowledge Distillation for Ukrainian Translation

***UCU NLP Course 2025***

## Task Overview
In this homework, you will implement knowledge distillation to compress a large teacher model (MamayLM-Gemma-3-4B-IT) into a smaller student model (Gemma-3-270M-IT) for Englishs to Ukrainian translation task.

**Teacher Model**: [INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0](https://huggingface.co/INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0) (4B parameters)  
**Student Model**: [google/gemma-3-270m-it](https://huggingface.co/google/gemma-3-270m-it) (270M parameters)  
**Dataset**: [lapa-llm/fiftyfive-best](https://huggingface.co/datasets/lapa-llm/fiftyfive-best)

**Note:** you may experiment with other student models, such as [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) with LoRA adapter on top, either using a standard [PEFT](https://huggingface.co/docs/peft/en/developer_guides/lora) package, or [Unsloth](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma3_(4B).ipynb).

**Note 2:** we do not use [Lapa LLM](https://huggingface.co/lapa-llm/lapa-v0.1.2-instruct) not only because of its size, but also a significantly modified vocabulary, which wouldn't allow us to compare logits directly.

## Setup and Installation

In [ ]:
# !pip install -q transformers datasets accelerate torch trl sacrebleu wandb

In [ ]:
from typing import Dict, List, Optional, Union, Any
from tqdm.auto import tqdm
import warnings

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
)

warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Load and Process Dataset

We'll use the `lapa-llm/fiftyfive-best` dataset which contains English-Ukrainian translation pairs.

**Note**: For faster experimentation, consider downsampling the dataset (e.g., use only 10k-50k examples for training).

In [ ]:
# Load the dataset
print("Loading dataset...")
dataset = load_dataset("lapa-llm/fiftyfive-best", split="train")

# For faster experimentation on Colab, downsample the dataset:
dataset = dataset.select(range(50_000))

print(f"Dataset size: {len(dataset)}")
print(f"Dataset features: {dataset.features}")
print("\nSample entry:")
print(dataset[0])

In [ ]:
# Load tokenizer (we'll use the student model's tokenizer for both models)
# Both models use the same Gemma tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-270m-it",
    # token="<your token here>",
)
    
print(f"Vocab size: {len(tokenizer)}")

Make sure you have padding token added. We are going to need it to train on batches with different sequence lengths. You may notice that many LLMs do not have a padding token, because they are trained using [Sample Packing](https://docs.axolotl.ai/docs/multipack.html), which both increases throughput and reduces memory requirements. We do not use this technique to avoid implementation overhead.

In [ ]:
# Gemma models need special tokens
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

### Data Preprocessing for Completion-Only Training

For instruction-tuned models, we want to compute loss only on the completion (translation output), not on the instruction/prompt. We'll set labels to -100 for the instruction tokens.

In [ ]:
def create_translation_prompts(example):
    """
    Create translation prompts in both directions.
    Assumes the dataset has 'en' and 'uk' fields.
    """

    # English to Ukrainian
    messages = [
        {
            "role": "user",
            "content": f"Translate the following text from English to Ukrainian:\n\n{example['en']}"
        },
    ]
    completion = example['uk']

    return {
        "messages": messages,
        "completion": completion
    }

In [ ]:
# Apply prompt creation
dataset = dataset.map(create_translation_prompts, num_proc=4)
print("Sample prompt:")
print(dataset[0]['messages'])
print(f"\nCompletion: {dataset[0]['completion']}")

***Preprocessing Task***

Implement a function to tokenize translation prompts for completion-only training in instruction-following language models.

The function should:

1. Apply chat template to the input (note: `add_generation_prompt=True`, see the [reason](https://huggingface.co/docs/transformers/main/en/chat_templating#addgenerationprompt)).
2. Tokenize the instruction.
3. Prepare labels:
     - For all instruction tokens, set the label to -100 (these will not contribute to the loss).
     - For completion tokens (including EOS), the label should be the token ID itself (these will contribute to the loss).
     - The final `labels` sequence should have shape identical to `input_ids`.
4.  Return a dictionary with the following fields (make sure the `max_length` is respected):
     - `"input_ids"`: token ids for the input,
     - `"attention_mask"`: mask for attention,
     - `"labels"`: contains -100 for instruction tokens and token ids for completion & EOS.


In [ ]:
def tokenize_with_completion_only(example, max_length=128):
    """
    Tokenize the conversation and set labels to -100 for instruction tokens.
    Only the completion part will contribute to the loss.
    """

    raise NotImplementedError("Not implemented")
    
    # input_ids = ...
    # attention_mask = ...
    # labels = ...
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [ ]:
# Apply tokenization
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    lambda x: tokenize_with_completion_only(x, max_length=128),
    remove_columns=dataset.column_names,
    num_proc=4,
    desc="Tokenizing"
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")
print(f"\nSample tokenized entry:")
sample = tokenized_dataset[0]
print(f"Input IDs length: {len(sample['input_ids'])}")
print(f"Number of training tokens (labels != -100): {sum(1 for l in sample['labels'] if l != -100)}")
print(f"\nDecoded input:\n{tokenizer.decode(sample['input_ids'])}")
print(f"\nDecoded labels (completion only):")
label_tokens = [t for t in sample['labels'] if t != -100]
print(tokenizer.decode(label_tokens))

## Prepare Teacher and Student Models

We load both models in `torch.bfloat16` precision, since they were originally trained using this precision. Besides, we consistently use `sdpa` (scaled dot product attention) attention implementation throughout the notebook.

In [ ]:
# Load teacher model (4B)
print("Loading teacher model (MamayLM 4B)...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    "INSAIT-Institute/MamayLM-Gemma-3-4B-IT-v1.0",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
teacher_model.eval()  # Set to eval mode - we won't train the teacher

print(f"Teacher model loaded. Parameters: {teacher_model.num_parameters() / 1e9:.2f}B")

In [ ]:
# Load student model (270M)
print("Loading student model (Gemma 270M)...")
student_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # token="<your token here>",
)

# Enable gradient checkpointing to save memory during training
# student_model.gradient_checkpointing_enable()

print(f"Student model loaded. Parameters: {student_model.num_parameters() / 1e6:.2f}M")

In [ ]:
# Ensure both models use the same pad token id
teacher_model.config.pad_token_id = tokenizer.pad_token_id
student_model.config.pad_token_id = tokenizer.pad_token_id

MamayLM (based on Gemma 3) has a vocabulary size of 262,208. Out of that, only 262,145 are real tokens, the rest are phantom, making sure the embedding layer dimension is a multiple of 64. 262,145th token corresponds to `<image_soft_token>` (MamayLM supports vision modality). However, a smaller Gemma 3 270m is a text-only LLM, and thus has a slightly reduced vocabulary. We need to account for that, so we resize token embeddings of the teacher model to be the same as the student's.

In [ ]:
vocab_size = student_model.config.get_text_config().vocab_size
teacher_model.config.get_text_config().vocab_size = vocab_size
teacher_model.resize_token_embeddings(vocab_size)

## Implement Distillation Trainer

***Training Task***

Implement `compute_loss` method of a custom trainer that inherits from HuggingFace's `Trainer` class.

It should:

1. Perform a forward pass on both models.
2. Calculate cross-entropy loss.
3. Calculate KL Divergence (distillation loss).
4. Aggregate them and return in an appropriate format.

Feel free and even **encouraged** to go through the DistilBERT's implementation at [transformers-research-projects/distillation](https://github.com/huggingface/transformers-research-projects/tree/main/distillation).

**Note on Trainer vs SFTTrainer**:
- `Trainer`: More flexible, full control over loss computation. Good when you need custom loss functions (like distillation).
- `SFTTrainer` (from [TRL](https://huggingface.co/docs/trl/main/en/sft_trainer)): Specialized for supervised fine-tuning with nice utilities for instruction datasets. However, harder to customize for distillation.

**For this task, we use `Trainer`** because we need custom loss computation (distillation + CE).

In [ ]:
class DistillationTrainer(Trainer):
    """
    Custom Trainer for Knowledge Distillation.
    
    Combines:
    1. Cross-Entropy loss (student predictions vs ground truth labels)
    2. KL-Divergence loss (student logits vs teacher logits)
    
    Args:
        teacher_model: The pre-trained teacher model (frozen)
        temperature: Temperature for softening probability distributions
        alpha: Weight for the distillation loss (1-alpha for CE loss)
    """
    
    def __init__(
        self,
        teacher_model,
        temperature: float = 2.0,
        ce_alpha: float = 2.0,
        kl_alpha: float = 5.0,
        *args,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.teacher_model.eval()  # Teacher is always in eval mode
        self.temperature = temperature
        self.ce_alpha = ce_alpha
        self.kl_alpha = kl_alpha
        
        # Move teacher to same device as student
        if hasattr(self.model, 'device'):
            self.teacher_model.to(self.model.device)
    
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        """
        Compute combined distillation and cross-entropy loss.
        """

        raise NotImplementedError("Not implemented")

        # ce_loss = ...
        # kl_loss = ...
        # loss = ...
        
        # Log individual losses for monitoring
        if self.state.global_step % 10 == 0:
            self.log({
                "ce_loss": ce_loss.item(),
                "kl_loss": kl_loss.item(),
                "combined_loss": loss.item()
            })
        
        return (loss, student_outputs) if return_outputs else loss

## Training Configuration

Set up training arguments and data collator.

We use `DataCollatorForTokenClassification`, because we already have `labels` prepared before to showcase the process. In practice, you could use [`DataCollatorForLanguageModeling`](https://huggingface.co/docs/transformers/v5.0.0rc1/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling) or its [TRL](https://huggingface.co/docs/trl/main/en/index) version ([`DataCollatorForLanguageModeling`](https://huggingface.co/docs/trl/v0.24.0/en/sft_trainer#trl.trainer.sft_trainer.DataCollatorForLanguageModeling)).

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    padding='longest',
)

Feel free to experiment with hyperparameters. We also use [W&B](https://wandb.ai/) to monitor the experiments (you may change that to whatever serves you best).

In [ ]:
training_args = TrainingArguments(
    output_dir="./distillation_output",
    
    # Training hyperparameters
    num_train_epochs=1,
    per_device_train_batch_size=2,  # Reduce if OOM
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    
    # Optimizer
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    
    # Performance
    bf16=True,  # Use bfloat16 for training
    # gradient_checkpointing=True,
    
    # Other
    report_to="wandb",  # Change to "wandb" if you want to use W&B
    remove_unused_columns=False,  # Important for custom data
)

## Train the Student Model

Now we'll train the student model using knowledge distillation. You can change the dataset size if the training takes too long.


In [ ]:
trainer = DistillationTrainer(
    teacher_model=teacher_model,
3
    model=student_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


In [ ]:
train_result = trainer.train()

In [ ]:
# Save the trained student model
trainer.save_model("./distilled_student_model")

## Evaluation

We'll evaluate all three models on the held-out subset:

1. Teacher model (MamayLM 4B)
2. Untrained student model (Gemma 270M IT baseline)
3. Distilled student model (our trained model)

The evaluation is implemented using `sacrebleu` to ensure a consistent way to compute the [BLEU metric](https://medium.com/nlplanet/two-minutes-nlp-learn-the-bleu-metric-by-examples-df015ca73a86).

In [ ]:
def evaluate_translation_manual(model, tokenizer, test_samples, max_samples=100):
    """
    Manually evaluate translation quality on FLORES-like test data.
    
    Args:
        model: The model to evaluate
        tokenizer: Tokenizer
        test_samples: Test dataset samples
        max_samples: Maximum samples to evaluate
    """
    from sacrebleu.metrics import BLEU, CHRF
    
    model.eval()
    
    predictions = []
    references = []
    
    print(f"Evaluating English to Ukrainian translation on {min(max_samples, len(test_samples))} samples...")
    
    for i, sample in enumerate(tqdm(test_samples.take(max_samples))):
        source = sample['en']
        target = sample['uk']
        prompt = f"Translate the following text from English to Ukrainian:\n\n{source}"
        
        messages = [{"role": "user", "content": prompt}]
        
        # Format and tokenize
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        # Decode
        generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        predictions.append(generated.strip())
        references.append(target)
    
    # Calculate BLEU and chrF scores
    bleu = BLEU()
    chrf = CHRF()
    
    bleu_score = bleu.corpus_score(predictions, [references])
    chrf_score = chrf.corpus_score(predictions, [references])
    
    print(f"\nResults for EN -> UK:")
    print(f"  BLEU: {bleu_score.score:.2f}")
    print(f"  chrF: {chrf_score.score:.2f}")
    
    def unfinished(text, cutoff: int = 200):
        return text[:cutoff] + '...' if len(text) > cutoff else text

    # Show some examples
    print("\nSample translations:")
    for i in range(min(10, len(predictions))):
        print(f"\nExample {i+1}:")
        print(f"  Source (EN): {unfinished(test_samples[i]['en'], cutoff=200)}")
        print(f"  Prediction: {unfinished(predictions[i], cutoff=200)}")
        print(f"  Reference: {unfinished(references[i], cutoff=200)}")
    
    return {
        "bleu": bleu_score.score,
        "chrf": chrf_score.score,
        "predictions": predictions,
        "references": references
    }


In [ ]:
# Last 1000 samples
test_dataset = load_dataset("lapa-llm/fiftyfive-best", split="train[-1000:]")

### Run Evaluations


In [ ]:
# Prepare results dictionary
results = {}

# Evaluate on English to Ukrainian
print("\n" + "="*60)
print("Evaluating: Teacher Model (MamayLM 4B)")
print("="*60)
results['teacher_en_uk'] = evaluate_translation_manual(
    teacher_model, tokenizer, test_dataset, max_samples=100
)

In [ ]:
# Evaluate baseline student (before distillation)
print("\n" + "="*60)
print("Evaluating: Baseline Student Model (Gemma 270M IT - Before Distillation)")
print("="*60)

baseline_student = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
baseline_student.eval()

results['baseline_en_uk'] = evaluate_translation_manual(
    baseline_student, tokenizer, test_dataset, max_samples=100
)

# Free memory
del baseline_student
torch.cuda.empty_cache()


In [ ]:
# Evaluate distilled student
print("\n" + "="*60)
print("Evaluating: Distilled Student Model (After Training)")
print("="*60)

# The student model is already trained
student_model.eval()

results['distilled_en_uk'] = evaluate_translation_manual(
    student_model, tokenizer, test_dataset, max_samples=100
)


## Results Comparison

Let's compare all the results in a nice table.


In [ ]:
import pandas as pd

# Create comparison table
comparison_data = [
    {
        "Model": "Teacher (MamayLM 4B)",
        "Parameters": "4B",
        "EN→UK BLEU": f"{results['teacher_en_uk']['bleu']:.2f}",
        "EN→UK chrF": f"{results['teacher_en_uk']['chrf']:.2f}",
    },
    {
        "Model": "Baseline Student (Gemma 270M)",
        "Parameters": "270M",
        "EN→UK BLEU": f"{results['baseline_en_uk']['bleu']:.2f}",
        "EN→UK chrF": f"{results['baseline_en_uk']['chrf']:.2f}",
    },
    {
        "Model": "Distilled Student (Gemma 270M)",
        "Parameters": "270M",
        "EN→UK BLEU": f"{results['distilled_en_uk']['bleu']:.2f}",
        "EN→UK chrF": f"{results['distilled_en_uk']['chrf']:.2f}",
    },
]

df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("FINAL RESULTS COMPARISON")
print("="*80 + "\n")
print(df.to_string(index=False))
print("\n" + "="*80)


### Using LM Evaluation Harness

For more robust evaluation with lm-eval on FLORES, you can:

```bash
lm_eval --model hf \
    --model_args pretrained=./distilled_student_model,dtype=bfloat16 \
    --tasks flores_en-uk \
    --batch_size auto \
    --include_path ./tasks \
    --log_samples \
    --apply_chat_template \
    --output_path ./eval_results
```
